# Phase 1 (Obj B Baseline): FE + IV Core Models

This notebook implements the baseline model layer for Obj B: panel FE and IV-TWFE estimates for inflation and GDP growth.

## Design choices

- Core controls: trade_open, pop_growth, investment_share.
- GDP models exclude gdp_pc_growth to avoid mechanical overlap risk.
- IV diagnostics are read from linearmodels first-stage diagnostics.
- Interpretation should separate relevance (chi2 significance) from strong-IV status.

In [ ]:
from pathlib import Path
import warnings
import pandas as pd
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[0] if (Path.cwd() / "macro_growth_merged.csv").exists() else Path.cwd().resolve()
if not (ROOT / "macro_growth_merged.csv").exists():
    ROOT = Path.cwd().resolve().parents[1]

OUT_DIR = ROOT / "outputs/notebook_phase1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESTRICTED_CONTROLS = ["trade_open", "pop_growth", "investment_share"]
CHI2_1_95_CRITICAL = 3.841458820694124
STRONG_IV_STAT_THRESHOLD = 10.0
ROOT

In [ ]:
base = pd.read_csv(ROOT / "macro_growth_merged.csv")
controls = pd.read_csv(ROOT / "data/phase1_controls.csv")
instruments = pd.read_csv(ROOT / "data/phase1_instruments.csv")

panel = (
    base.merge(controls, on=["Country Name", "year"], how="left")
        .merge(instruments, on=["Country Name", "year"], how="left")
        .rename(columns={"Country Name": "country"})
        .sort_values(["country", "year"])
        .copy()
)

pd.DataFrame({
    "metric": ["rows", "countries", "year_min", "year_max"],
    "value": [len(panel), panel["country"].nunique(), int(panel["year"].min()), int(panel["year"].max())],
})

In [ ]:
def run_fe(df, outcome, controls=None):
    controls = controls or []
    cols = ["country", "year", outcome, "m2_growth", *controls]
    fit_data = df[cols].dropna().set_index(["country", "year"]).sort_index()
    rhs = "m2_growth" + (" + " + " + ".join(controls) if controls else "")
    formula = f"{outcome} ~ 1 + {rhs} + EntityEffects + TimeEffects"
    return PanelOLS.from_formula(formula, data=fit_data).fit(cov_type="clustered", cluster_entity=True)

fe_infl_base = run_fe(panel, "inflation", controls=[])
fe_gdp_base = run_fe(panel, "gdp_growth", controls=[])
fe_infl_ctrl = run_fe(panel, "inflation", controls=RESTRICTED_CONTROLS)
fe_gdp_ctrl = run_fe(panel, "gdp_growth", controls=RESTRICTED_CONTROLS)

fe_table = pd.DataFrame([
    {"model": "fe_baseline", "outcome": "inflation", "coef": float(fe_infl_base.params["m2_growth"]), "p_value": float(fe_infl_base.pvalues["m2_growth"]), "nobs": int(fe_infl_base.nobs)},
    {"model": "fe_baseline", "outcome": "gdp_growth", "coef": float(fe_gdp_base.params["m2_growth"]), "p_value": float(fe_gdp_base.pvalues["m2_growth"]), "nobs": int(fe_gdp_base.nobs)},
    {"model": "fe_controls", "outcome": "inflation", "coef": float(fe_infl_ctrl.params["m2_growth"]), "p_value": float(fe_infl_ctrl.pvalues["m2_growth"]), "nobs": int(fe_infl_ctrl.nobs)},
    {"model": "fe_controls", "outcome": "gdp_growth", "coef": float(fe_gdp_ctrl.params["m2_growth"]), "p_value": float(fe_gdp_ctrl.pvalues["m2_growth"]), "nobs": int(fe_gdp_ctrl.nobs)},
])
fe_table

In [ ]:
def run_iv_twfe(df, outcome, instrument, controls):
    cols = ["country", "year", outcome, "m2_growth", instrument, *controls]
    fit_data = df[cols].dropna().copy()

    rhs = " + ".join(controls)
    if rhs:
        formula = f"{outcome} ~ 1 + {rhs} + C(country) + C(year) [m2_growth ~ {instrument}]"
    else:
        formula = f"{outcome} ~ 1 + C(country) + C(year) [m2_growth ~ {instrument}]"

    res = IV2SLS.from_formula(formula, data=fit_data).fit(cov_type="clustered", clusters=fit_data["country"])
    fs = res.first_stage.diagnostics.loc["m2_growth"]
    return {
        "outcome": outcome,
        "instrument": instrument,
        "coef": float(res.params["m2_growth"]),
        "p_value": float(res.pvalues["m2_growth"]),
        "nobs": int(res.nobs),
        "first_stage_stat": float(fs["f.stat"]),
        "first_stage_p": float(fs["f.pval"]),
        "partial_rsquared": float(fs["partial.rsquared"]),
    }

iv_rows = []
for outcome in ["inflation", "gdp_growth"]:
    for instrument in ["instrument_m2_external_level", "instrument_m2_l1"]:
        iv_rows.append(run_iv_twfe(panel, outcome, instrument, RESTRICTED_CONTROLS))

iv_table = pd.DataFrame(iv_rows).sort_values(["outcome", "instrument"])
iv_table

In [ ]:
preferred = iv_table[(iv_table["outcome"] == "inflation") & (iv_table["instrument"] == "instrument_m2_external_level")].iloc[0]
gate_table = pd.DataFrame([
    {"criterion": "first_stage_relevance_chi2_95", "value": float(preferred["first_stage_stat"]), "threshold": f">{CHI2_1_95_CRITICAL:.4f}", "pass": bool(preferred["first_stage_stat"] > CHI2_1_95_CRITICAL)},
    {"criterion": "first_stage_p_lt_0p05", "value": float(preferred["first_stage_p"]), "threshold": "<0.05", "pass": bool(preferred["first_stage_p"] < 0.05)},
    {"criterion": "first_stage_strong_iv_ge_10", "value": float(preferred["first_stage_stat"]), "threshold": f">={STRONG_IV_STAT_THRESHOLD:.1f}", "pass": bool(preferred["first_stage_stat"] >= STRONG_IV_STAT_THRESHOLD)},
])

fe_table.to_csv(OUT_DIR / "phase1_fe_results.csv", index=False)
iv_table.to_csv(OUT_DIR / "phase1_iv_results.csv", index=False)
gate_table.to_csv(OUT_DIR / "phase1_gate_read.csv", index=False)

display(gate_table)
print("Saved Phase 1 outputs to:", OUT_DIR)

## Interpretation for Obj B baseline

- Use FE and IV coefficient signs/magnitudes as baseline evidence.
- Keep causal language constrained unless first-stage strength is comfortably high.
- Continue to Phase 2 for short-run LP-IV dynamics.